# SKAB — Model Comparison

RF, IF, and MLP with three-fold file-level cross-validation. The baseline test split is reused.


In [ ]:
from pathlib import Path
from datetime import datetime
from itertools import product
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys,json,time,random,hashlib,urllib.request,shutil,platform,copy,importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier,IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,average_precision_score
import torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA = ROOT / 'data' / 'SKAB'
RUN = ROOT / 'results' / 'model_comparison' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
DATA.mkdir(parents=True,exist_ok=True);RUN.mkdir(parents=True,exist_ok=True)
(RUN/'checkpoints').mkdir(exist_ok=True)
COMMIT='b2c0d46c2971dcbfe71e26087b6d231998bb91c2'
SENSORS=['Accelerometer1RMS','Accelerometer2RMS','Current','Pressure','Temperature','Thermocouple','Voltage','Volume Flow RateRMS']
SEEDS=[0,1,2]; CV_SEED=42; SEARCH_SEED=0
WINDOW=30; TREES=200; N_JOBS=2
MLP_MAX_EPOCHS=60; MLP_PATIENCE=8; BATCH_SIZE=512
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(2)
plt.rcParams.update({'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False})
print('장치:',DEVICE,'| torch:',torch.__version__,'| sklearn:',sklearn.__version__)
print('결과 폴더:',RUN)

## 1. Dataset and Cross-Validation Splits


In [ ]:
files = ([f'valve1/{i}.csv' for i in range(16)]
         + [f'valve2/{i}.csv' for i in range(4)]
         + [f'other/{i}.csv' for i in range(1,15)])
records, tables, download_manifest = [], {}, []
def download_file(fid):
    dest = DATA / fid
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        url = f'https://raw.githubusercontent.com/waico/SKAB/{COMMIT}/data/{fid}'
        last_error = None
        for attempt in range(3):
            try:
                request = urllib.request.Request(url, headers={'User-Agent':'SKAB-course-project'})
                with urllib.request.urlopen(request, timeout=60) as response:
                    payload = response.read()
                tmp = dest.with_suffix('.download')
                tmp.write_bytes(payload)
                tmp.replace(dest)
                last_error = None
                break
            except Exception as exc:
                last_error = exc
        if last_error is not None:
            raise RuntimeError(f'{fid} 다운로드 실패. 인터넷 연결 확인 후 이 셀을 다시 실행하세요.') from last_error
    return fid

with ThreadPoolExecutor(max_workers=4) as pool:
    tasks = [pool.submit(download_file, fid) for fid in files]
    for j, future in enumerate(as_completed(tasks), 1):
        future.result()
        if j % 10 == 0 or j == len(files): print(f'다운로드 준비: {j}/{len(files)}')

for j, fid in enumerate(files, 1):
    dest = DATA / fid
    df = pd.read_csv(dest, sep=';')
    required = SENSORS + ['datetime','anomaly','changepoint']
    if not set(required).issubset(df.columns):
        raise ValueError(f'{fid}의 열이 예상과 다릅니다: {df.columns.tolist()}')
    df['datetime'] = pd.to_datetime(df['datetime'], errors='raise')
    df = df.sort_values('datetime', kind='stable').reset_index(drop=True)
    if df['datetime'].duplicated().any():
        raise ValueError(f'{fid}: 중복 시각을 먼저 확인해야 합니다.')
    for col in SENSORS:
        df[col] = pd.to_numeric(df[col], errors='raise')
    if not df.anomaly.isin([0,1]).all():
        raise ValueError(f'{fid}: 정답 라벨 확인 필요')
    tables[fid] = df
    records.append({'file':fid,'family':fid.split('/')[0], 'rows':len(df),
                    'anomalies':int(df.anomaly.sum()),'anomaly_rate':df.anomaly.mean(),
                    'missing_sensors':int(df[SENSORS].isna().sum().sum()),
                    'median_interval_seconds':df.datetime.diff().dt.total_seconds().median()})
    download_manifest.append({'file':fid, 'sha256':hashlib.sha256(dest.read_bytes()).hexdigest()})
    if j % 10 == 0 or j == len(files): print(f'데이터 준비: {j}/{len(files)}')
inventory = pd.DataFrame(records)
inventory.to_csv(RUN/'data_inventory.csv',index=False,encoding='utf-8-sig')
print('총 파일:', len(inventory), '| 총 행:', inventory.rows.sum())
print('이상 비율:', f'{inventory.anomalies.sum()/inventory.rows.sum():.2%}')
print('센서 결측값:', inventory.missing_sensors.sum())
display(inventory.groupby('family').agg(files=('file','count'),rows=('rows','sum'),anomalies=('anomalies','sum')))
display(inventory[['family','median_interval_seconds']].groupby('family').describe())
display(next(iter(tables.values())).head())

In [ ]:
TEST_IDS=['other/11.csv', 'other/4.csv', 'other/7.csv', 'valve1/1.csv', 'valve1/5.csv', 'valve1/7.csv', 'valve2/3.csv']
DEV_IDS=sorted(set(files)-set(TEST_IDS))
assert len(DEV_IDS)==27 and len(TEST_IDS)==7
assert set(TEST_IDS).isdisjoint(DEV_IDS)
folds=[]; fold_rows=[]
splitter=StratifiedKFold(n_splits=3,shuffle=True,random_state=CV_SEED)
families=[fid.split('/')[0] for fid in DEV_IDS]
for fold,(ti,vi) in enumerate(splitter.split(DEV_IDS,families),1):
    train_files=[DEV_IDS[i] for i in ti];validation_files=[DEV_IDS[i] for i in vi]
    assert set(train_files).isdisjoint(validation_files)
    folds.append((train_files,validation_files))
    for role,ids in [('train',train_files),('validation',validation_files)]:
        fold_rows.extend({'fold':fold,'role':role,'file':fid,'family':fid.split('/')[0]} for fid in ids)
    print(f'Fold {fold}: 학습 {len(train_files)}파일 / 검증 {len(validation_files)}파일')
pd.DataFrame(fold_rows).to_csv(RUN/'cv_file_splits.csv',index=False,encoding='utf-8-sig')
inventory['role']=inventory.file.map(lambda f:'test' if f in TEST_IDS else 'development')
inventory.to_csv(RUN/'development_test_split.csv',index=False,encoding='utf-8-sig')
display(inventory.groupby('role').agg(files=('file','count'),rows=('rows','sum'),anomalies=('anomalies','sum')))
print('테스트 파일:',TEST_IDS)

## 2. Feature Engineering and Preprocessing


In [ ]:
def features(df,mode):
    x=df[SENSORS].astype(float).replace([np.inf,-np.inf],np.nan)
    if mode=='raw':return x.copy()
    mean=x.rolling(WINDOW,min_periods=1).mean().add_suffix('_mean30')
    std=x.rolling(WINDOW,min_periods=1).std(ddof=0).add_suffix('_std30')
    delta=x.diff().add_suffix('_diff1');delta.iloc[0]=0.
    return pd.concat([x,mean,std,delta],axis=1)

def assemble(ids,mode):
    xx=[];yy=[];mm=[]
    for fid in sorted(ids):
        df=tables[fid]
        xx.append(features(df,mode));yy.append(df.anomaly.to_numpy(dtype=int))
        mm.append(pd.DataFrame({'file':fid,'row':np.arange(len(df)),'datetime':df.datetime.astype(str)}))
    return pd.concat(xx,ignore_index=True),np.concatenate(yy),pd.concat(mm,ignore_index=True)

cv_packs={}
for mode in ['raw','temporal']:
    for fi,(tr,va) in enumerate(folds,1):
        xtr,ytr,_=assemble(tr,mode);xva,yva,_=assemble(va,mode)
        imp=SimpleImputer(strategy='median',keep_empty_features=True)
        a=imp.fit_transform(xtr).astype('float32');b=imp.transform(xva).astype('float32')
        scaler=StandardScaler().fit(a)
        cv_packs[(mode,fi)]={'train':a,'val':b,'ytrain':ytr,'yval':yva,
                            'mlp_train':scaler.transform(a).astype('float32'),
                            'mlp_val':scaler.transform(b).astype('float32')}
        assert set(np.unique(ytr))=={0,1} and set(np.unique(yva))=={0,1}
print('교차검증 입력 준비 완료. 테스트 입력은 설정 선택 이후에 준비합니다.')

## 3. Models and Hyperparameter Search Space


In [ ]:
def seed_everything(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True

class MLP(nn.Module):
    def __init__(self,n_features,hidden):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(n_features,hidden[0]),nn.ReLU(),
                               nn.Linear(hidden[0],hidden[1]),nn.ReLU(),nn.Linear(hidden[1],1))
    def forward(self,x):return self.net(x).squeeze(-1)

def fit_mlp(x,y,params,seed,xval=None,yval=None,epochs=None):
    seed_everything(seed)
    net=MLP(x.shape[1],params['hidden']).to(DEVICE)
    # 클래스 불균형을 고려한 가중 BCE. RF도 class_weight='balanced' 사용.
    pos_weight=torch.tensor(float((y==0).sum()/max(1,(y==1).sum())),device=DEVICE)
    criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer=torch.optim.AdamW(net.parameters(),lr=params['lr'],weight_decay=1e-4)
    generator=torch.Generator().manual_seed(seed)
    loader=DataLoader(TensorDataset(torch.from_numpy(x),torch.from_numpy(y.astype('float32'))),
                      batch_size=BATCH_SIZE,shuffle=True,generator=generator,num_workers=0)
    has_val=xval is not None
    if has_val:
        vx=torch.from_numpy(xval).to(DEVICE);vy=torch.from_numpy(yval.astype('float32')).to(DEVICE)
    best_loss=np.inf;best_state=None;best_epoch=0;stale=0;history=[]
    limit=MLP_MAX_EPOCHS if epochs is None else int(epochs)
    for epoch in range(1,limit+1):
        net.train();total=0.
        for xb,yb in loader:
            xb=xb.to(DEVICE);yb=yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss=criterion(net(xb),yb);loss.backward();optimizer.step()
            total+=float(loss.detach().cpu())*len(yb)
        row={'epoch':epoch,'train_bce':total/len(y)}
        if has_val:
            net.eval()
            with torch.no_grad():loss_val=float(criterion(net(vx),vy).cpu())
            row['validation_bce']=loss_val
            if loss_val<best_loss-1e-5:
                best_loss=loss_val;best_state={k:v.detach().cpu().clone() for k,v in net.state_dict().items()}
                best_epoch=epoch;stale=0
            else:stale+=1
        history.append(row)
        if has_val and stale>=MLP_PATIENCE:break
    if has_val:net.load_state_dict(best_state)
    else:best_epoch=limit
    net.eval()
    return net,best_epoch,history

def score(model,kind,x):
    if kind=='RF':return model.predict_proba(x)[:,list(model.classes_).index(1)]
    if kind=='IF':return -model.score_samples(x)
    out=[];model.eval()
    with torch.no_grad():
        for start in range(0,len(x),4096):
            out.append(torch.sigmoid(model(torch.from_numpy(x[start:start+4096]).to(DEVICE))).cpu().numpy())
    return np.concatenate(out)

def fit_tree(kind,x,y,params,seed):
    if kind=='RF':
        model=RandomForestClassifier(n_estimators=TREES,random_state=seed,n_jobs=N_JOBS,
                                      class_weight='balanced',**params)
        model.fit(x,y)
    else:
        model=IsolationForest(n_estimators=TREES,random_state=seed,n_jobs=N_JOBS,contamination='auto',**params)
        model.fit(x[y==0])
    return model

def measures(y,scores,cut):
    pred=(scores>=cut).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),
            'f1':f1_score(y,pred,zero_division=0),'pr_auc_ap':average_precision_score(y,scores),
            'false_positive_rate':fp/(fp+tn) if fp+tn else 0.,
            'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}

GRIDS={'RF':[{'max_depth':d,'min_samples_leaf':l} for d,l in product([12,None],[1,5])],
       'IF':[{'max_samples':s} for s in [256,1024]],
       'MLP':[{'hidden':list(h),'lr':lr} for h,lr in product([(64,32),(128,64)],[.001,.0003])]}
PROB_CUTS=[round(float(v),2) for v in np.arange(.1,1.,.1)]
NORMAL_QUANTILES=PROB_CUTS+[.95,.99]
print('MLP 구조 예시: 입력 → 64 → 32 → 출력1')
print('탐색할 fold 학습 횟수:',sum(len(v) for v in GRIDS.values())*2*len(folds))

## 4. Cross-Validation and Model Selection


In [ ]:
cv_rows=[];loss_rows=[];done=0
total=sum(len(v) for v in GRIDS.values())*2*len(folds)
tic=time.perf_counter()
for kind in ['RF','IF','MLP']:
    for mode in ['raw','temporal']:
        for ci,params in enumerate(GRIDS[kind]):
            for fi in range(1,4):
                cache=RUN/'checkpoints'/f'cv_{kind}_{mode}_{ci}_{fi}.json'
                if cache.exists():
                    saved=json.loads(cache.read_text());rows=saved['rows'];history=saved['history']
                else:
                    p=cv_packs[(mode,fi)];tick=time.perf_counter()
                    if kind=='MLP':
                        model,ep,history=fit_mlp(p['mlp_train'],p['ytrain'],params,SEARCH_SEED,p['mlp_val'],p['yval'])
                        scores=score(model,kind,p['mlp_val'])
                    else:
                        model=fit_tree(kind,p['train'],p['ytrain'],params,SEARCH_SEED)
                        scores=score(model,kind,p['val']);ep=0;history=[]
                    candidates=NORMAL_QUANTILES if kind=='IF' else PROB_CUTS
                    normal_scores=score(model,kind,p['train'][p['ytrain']==0]) if kind=='IF' else None
                    rows=[]
                    for level in candidates:
                        cut=float(np.quantile(normal_scores,level)) if kind=='IF' else level
                        rows.append({'model':kind,'features':mode,'config_id':ci,'fold':fi,
                                     'params':json.dumps(params,sort_keys=True),'threshold_level':level,
                                     'threshold_kind':'training_normal_quantile' if kind=='IF' else 'probability',
                                     'actual_threshold':cut,'best_epoch':ep,
                                     **measures(p['yval'],scores,cut)})
                    cache.write_text(json.dumps({'rows':rows,'history':history}),encoding='utf-8')
                    del model
                    if DEVICE.type=='cuda':torch.cuda.empty_cache()
                cv_rows.extend(rows)
                loss_rows.extend({**h,'model':kind,'features':mode,'config_id':ci,'fold':fi} for h in history)
                done+=1
                local_best=max(rows,key=lambda r:r['f1'])
                print(f"[{done:02d}/{total}] {kind:3s} {mode:8s} 설정{ci} fold{fi} | 후보 최고 F1 {local_best['f1']:.3f} | 누적 {(time.perf_counter()-tic)/60:.1f}분",flush=True)
                pd.DataFrame(cv_rows).to_csv(RUN/'cv_trials.csv',index=False,encoding='utf-8-sig')
cv_detail=pd.DataFrame(cv_rows)
if loss_rows:pd.DataFrame(loss_rows).to_csv(RUN/'mlp_cv_learning_curves.csv',index=False,encoding='utf-8-sig')
keys=['model','features','config_id','params','threshold_level','threshold_kind']
cv_summary=cv_detail.groupby(keys,dropna=False).agg(
    f1_mean=('f1','mean'),f1_std=('f1','std'),precision_mean=('precision','mean'),
    recall_mean=('recall','mean'),ap_mean=('pr_auc_ap','mean'),folds=('fold','nunique')).reset_index()
assert cv_summary.folds.eq(3).all()
cv_summary=cv_summary.sort_values(['f1_mean','ap_mean','threshold_level'],ascending=False,kind='stable')
cv_summary.to_csv(RUN/'cv_search_summary.csv',index=False,encoding='utf-8-sig')
selected=cv_summary.drop_duplicates(['model','features']).copy()
selected.to_csv(RUN/'selected_configurations.csv',index=False,encoding='utf-8-sig')
SELECTED_MODEL=str(selected.iloc[0]['model']);SELECTED_FEATURES=str(selected.iloc[0]['features'])
final_specs=[]
for r in selected.to_dict('records'):
    params=json.loads(r['params'])
    epoch_rows=cv_detail[(cv_detail.model==r['model'])&(cv_detail.features==r['features'])&(cv_detail.config_id==r['config_id'])]
    epochs=int(np.ceil(epoch_rows.groupby('fold').best_epoch.first().median())) if r['model']=='MLP' else 0
    final_specs.append({'model':r['model'],'features':r['features'],'params':params,
                        'threshold_level':float(r['threshold_level']),'epochs':epochs,'config_id':int(r['config_id'])})
(RUN/'frozen_selection.json').write_text(json.dumps({'specifications':final_specs,'selected_model':SELECTED_MODEL,'selected_features':SELECTED_FEATURES},indent=2),encoding='utf-8')
print('테스트 확인 전 선택:',SELECTED_MODEL,SELECTED_FEATURES)
display(selected[['model','features','f1_mean','f1_std','threshold_level','params']].round(4))
print('MLP 최종 학습 epoch는 fold별 선택 epoch의 중앙값(올림)으로 고정합니다.')

## 5. Final Training and Test Evaluation


In [ ]:
final_packs={}
for mode in ['raw','temporal']:
    xd,yd,mdv=assemble(DEV_IDS,mode);xt,yt,mt=assemble(TEST_IDS,mode)
    imp=SimpleImputer(strategy='median',keep_empty_features=True)
    a=imp.fit_transform(xd).astype('float32');b=imp.transform(xt).astype('float32')
    scaler=StandardScaler().fit(a)
    final_packs[mode]={'train':a,'test':b,'ytrain':yd,'ytest':yt,'meta':mt,
                        'mlp_train':scaler.transform(a).astype('float32'),
                        'mlp_test':scaler.transform(b).astype('float32')}

def evaluate_job(spec,seed,label='optimized'):
    kind=spec['model'];mode=spec['features'];p=final_packs[mode]
    stem=f'final_{label}_{kind}_{mode}_{seed}'
    cache=RUN/'checkpoints'/(stem+'.json');pred_cache=RUN/'checkpoints'/(stem+'.csv.gz')
    if cache.exists() and pred_cache.exists():
        return json.loads(cache.read_text())['rows'],pd.read_csv(pred_cache)
    tick=time.perf_counter()
    if kind=='MLP':
        model,ep,history=fit_mlp(p['mlp_train'],p['ytrain'],spec['params'],seed,epochs=spec['epochs'])
        scores=score(model,kind,p['mlp_test'])
        pd.DataFrame(history).to_csv(RUN/(stem+'_loss.csv'),index=False)
    else:
        model=fit_tree(kind,p['train'],p['ytrain'],spec['params'],seed)
        scores=score(model,kind,p['test'])
    if kind=='IF':
        normal_scores=score(model,kind,p['train'][p['ytrain']==0])
        cut=float(np.quantile(normal_scores,spec['threshold_level']))
        default_cut=float(np.quantile(normal_scores,.99))
    else:cut=spec['threshold_level'];default_cut=.5
    out=[]
    rules=[('cv_selected',cut),('default',default_cut)] if label=='optimized' else [('fixed_control',.5)]
    for rule,c in rules:
        out.append({'model':kind,'features':mode,'experiment':label,'seed':seed,'rule':rule,
                     'actual_threshold':float(c),'threshold_level':spec['threshold_level'],
                     'params':json.dumps(spec['params'],sort_keys=True),
                     **measures(p['ytest'],scores,c)})
    pred=p['meta'].copy();pred['y_true']=p['ytest'];pred['score']=scores
    pred['prediction']=(scores>=cut).astype(int);pred['default_prediction']=(scores>=default_cut).astype(int)
    pred['model']=kind;pred['features']=mode;pred['seed']=seed;pred['experiment']=label
    pred.to_csv(pred_cache,index=False,compression='gzip')
    cache.write_text(json.dumps({'rows':out}),encoding='utf-8')
    print(f'{kind} {mode} seed{seed} {label} 완료 ({time.perf_counter()-tick:.1f}초)',flush=True)
    del model
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
    return out,pred

test_rows=[];predictions=[]
for spec in final_specs:
    for seed in SEEDS:
        rows,pred=evaluate_job(spec,seed)
        test_rows.extend(rows);predictions.append(pred)
test_detail=pd.DataFrame(test_rows)
test_detail.to_csv(RUN/'test_metrics_all_rules.csv',index=False,encoding='utf-8-sig')
metric_cols=['precision','recall','f1','pr_auc_ap','false_positive_rate']
def summarize(df,group_cols):
    result=df.groupby(group_cols)[metric_cols].agg(['mean','std'])
    result.columns=['_'.join(c) for c in result.columns]
    return result.reset_index()
test_summary=summarize(test_detail.query("rule=='cv_selected'"),['model','features'])
test_summary.to_csv(RUN/'test_summary.csv',index=False,encoding='utf-8-sig')
summarize(test_detail,['model','features','rule']).to_csv(RUN/'threshold_comparison.csv',index=False,encoding='utf-8-sig')
print('최종 테스트 결과 (선택은 교차검증으로 확정):')
display(test_summary.round(4))
print('선택 유지:',SELECTED_MODEL,SELECTED_FEATURES)

control_rows=[];control_predictions=[]
reference={'max_depth':12,'min_samples_leaf':1}
for mode in ['raw','temporal']:
    matching=next(s for s in final_specs if s['model']=='RF' and s['features']==mode)
    for seed in SEEDS:
        if matching['params']==reference:
            rr=test_detail[(test_detail.model=='RF')&(test_detail.features==mode)&(test_detail.seed==seed)&(test_detail.rule=='default')].iloc[0].to_dict()
            rr.update(experiment='controlled_input',rule='fixed_control',threshold_level=.5)
            pp=next(p for p in predictions if p.model.iloc[0]=='RF' and p.features.iloc[0]==mode and p.seed.iloc[0]==seed).copy()
            pp['prediction']=pp.default_prediction;pp['experiment']='controlled_input'
            control_rows.append(rr);control_predictions.append(pp)
        else:
            spec={'model':'RF','features':mode,'params':reference,'threshold_level':.5,'epochs':0}
            rr,pp=evaluate_job(spec,seed,'controlled_input')
            control_rows.extend(rr);control_predictions.append(pp)
control_detail=pd.DataFrame(control_rows)
control_detail.to_csv(RUN/'controlled_input_metrics.csv',index=False,encoding='utf-8-sig')
control_summary=summarize(control_detail,['model','features'])
control_summary.to_csv(RUN/'controlled_input_summary.csv',index=False,encoding='utf-8-sig')
print('RF 설정·임계값을 고정한 입력 비교:')
display(control_summary.round(4))
prediction_all=pd.concat(predictions+control_predictions,ignore_index=True)
prediction_all.to_csv(RUN/'test_predictions.csv.gz',index=False,compression='gzip')

## 6. Visualization and File-Level Evaluation


In [ ]:
fig,ax=plt.subplots(figsize=(9,4))
order=['RF','IF','MLP'];width=.34;x=np.arange(3)
for offset,mode,color in [(-width/2,'raw','#1E3A5F'),(width/2,'temporal','#4A90D9')]:
    rr=test_summary.set_index(['model','features']).loc[[(k,mode) for k in order]]
    ax.bar(x+offset,rr.f1_mean,width,yerr=rr.f1_std,capsize=3,label=mode,color=color)
ax.set_xticks(x,order);ax.set_ylim(0,1.05);ax.set_ylabel('Test F1');ax.legend()
ax.set_title('RF / IF / MLP comparison');fig.tight_layout()
fig.savefig(RUN/'model_comparison.png',dpi=190);plt.show()

per_file=[]
for (kind,mode,seed,exp,fid),d in prediction_all.groupby(['model','features','seed','experiment','file']):
    y=d.y_true.to_numpy();pred=d.prediction.to_numpy()
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    per_file.append({'model':kind,'features':mode,'seed':seed,'experiment':exp,'file':fid,
        'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),
        'f1':f1_score(y,pred,zero_division=0),'pr_auc_ap':average_precision_score(y,d.score),
        'false_positive_rate':fp/(fp+tn),'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn)})
per_file=pd.DataFrame(per_file)
per_file.to_csv(RUN/'test_metrics_by_file.csv',index=False,encoding='utf-8-sig')
display(per_file.query("experiment=='optimized'").groupby(['file','model','features']).f1.mean().unstack(['model','features']).round(3))

fid=sorted(TEST_IDS)[0]
chosen=prediction_all.query("experiment=='optimized'")
chosen=chosen[(chosen.model==SELECTED_MODEL)&(chosen.features==SELECTED_FEATURES)&(chosen.seed==0)&(chosen.file==fid)].sort_values('row')
fig,axs=plt.subplots(2,1,figsize=(10,5),sharex=True)
axs[0].step(chosen.row,chosen.y_true,where='post',color='#1E3A5F');axs[0].set_ylabel('Actual anomaly')
axs[1].step(chosen.row,chosen.prediction,where='post',color='#4A90D9');axs[1].set_ylabel('Predicted anomaly')
for ax in axs:ax.set_ylim(-.1,1.1);ax.set_yticks([0,1])
axs[0].set_title(f'{SELECTED_MODEL} / {SELECTED_FEATURES} | {fid} | seed 0')
axs[1].set_xlabel('Observation index within experiment')
fig.tight_layout();fig.savefig(RUN/'selected_model_example.png',dpi=190);plt.show()

if loss_rows:
    losses=pd.DataFrame(loss_rows)
    mlp_spec=next(s for s in final_specs if s['model']=='MLP' and s['features']==SELECTED_FEATURES)
    d=losses[(losses.features==SELECTED_FEATURES)&(losses.config_id==mlp_spec['config_id'])&(losses.fold==1)]
    fig,ax=plt.subplots(figsize=(7,3.5))
    ax.plot(d.epoch,d.train_bce,label='train');ax.plot(d.epoch,d.validation_bce,label='validation')
    ax.set_xlabel('Epoch');ax.set_ylabel('Weighted BCE');ax.legend();fig.tight_layout()
    fig.savefig(RUN/'mlp_learning_curve.png',dpi=180);plt.show()

## 7. Export Results


In [ ]:
manifest={'dataset':'SKAB v0.9 labeled 34 files','source':'https://github.com/waico/SKAB','commit':COMMIT,
 'development_ids':DEV_IDS,'test_ids':TEST_IDS,'same_test_as_previous':True,'data_sha256':download_manifest,
 'cv':{'n_splits':3,'split_seed':CV_SEED,'unit':'whole experiment files','stratification':'coarse family at file level',
       'selection':'mean fold pointwise F1, tie AP then higher threshold level','search_seed':SEARCH_SEED},
 'seeds_final':SEEDS,'window_observations':WINDOW,'trees':TREES,'grids':GRIDS,
 'probability_thresholds':PROB_CUTS,'if_training_normal_quantiles':NORMAL_QUANTILES,
 'mlp':{'activation':'ReLU','loss':'BCEWithLogitsLoss with train-derived pos_weight','optimizer':'AdamW',
        'weight_decay':1e-4,'batch_size':BATCH_SIZE,'max_epochs':MLP_MAX_EPOCHS,'patience':MLP_PATIENCE,
        'cv_epoch_selection':'minimum fold-validation weighted BCE; early stopping',
        'final_epochs':'ceil median of best CV epochs; no test monitoring','scaling':'fold-train StandardScaler'},
 'selected_by_cv':{'model':SELECTED_MODEL,'features':SELECTED_FEATURES},'final_specs':final_specs,
 'label_usage':{'RF':'train anomaly labels for classification, CV labels for tuning',
                'IF':'train anomaly labels to select normal rows, CV labels for tuning',
                'MLP':'train anomaly labels for classification, CV labels for epoch and setting selection'},
 'controlled_input':{'model':'RF','params':reference,'threshold':.5,'same_development_data_and_seeds':True},
 'point_adjustment':False,'pr_auc_definition':'sklearn average_precision_score (AP)',
 'limitations':['test results were already observed before this follow-up; not a newly untouched test',
  'CV used for tuning, not an unbiased final performance estimate','one fixed 7-file test set',
  'MLP search budget 4, RF 4, IF 2; not identical model family search spaces',
  'IF uses fewer fitting rows (normal only); not equal label-information comparison',
  '27 development files for final fitting vs 20 training files in previous experiment',
  'seed standard deviation measures optimization randomness, not uncertainty across test splits'],
 'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
             'sklearn':sklearn.__version__,'torch':str(torch.__version__),'device':str(DEVICE)}}
(RUN/'experiment_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
(RUN/'READ_ME.txt').write_text(chr(10).join([
 '설정 선택: selected_configurations.csv / CV전체: cv_search_summary.csv',
 '최종 성능: test_summary.csv / 임계값비교: threshold_comparison.csv',
 '입력만 변경한 RF 비교: controlled_input_summary.csv',
 '실험별 결과: test_metrics_by_file.csv / 원본 예측: test_predictions.csv.gz',
 '이번 최종 학습은 개발27파일이며 이전20파일 실험과 직접 개선량을 비교하지 않습니다.'
 ]),encoding='utf-8-sig')
import zipfile
zip_path=RUN.parent/(RUN.name+'_cv_mlp_results.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RUN.iterdir()):
        if p.is_file():z.write(p,arcname=p.name)
print('보완 실험 완료!')
print('교차검증에서 선택:',SELECTED_MODEL,SELECTED_FEATURES)
print('Results archive:', zip_path)
print(f'크기: {zip_path.stat().st_size/1024/1024:.2f} MB')

In [ ]:
try:
    from google.colab import files as colab_files
except ImportError:
    print('Results archive:', zip_path)
else:
    colab_files.download(str(zip_path))